In [1]:
!pip install biopython

In [2]:
import os
import pandas as pd
from Bio.PDB import MMCIFParser
from math import sqrt
import itertools
import json

In [3]:
def calculate_distance(atom1, atom2):
    diff = atom1.coord - atom2.coord
    return sqrt((diff ** 2).sum())

In [4]:
def get_distances(chain_A, chain_B, dock_label):
    distances = []
    for atom_A, atom_B in itertools.product(chain_A.get_atoms(), chain_B.get_atoms()):
        dist = calculate_distance(atom_A, atom_B)
        distances.append({
            "atom1_name": atom_A.get_name(),             # Atom name, e.g., "CG2"
            "atom1_residue": atom_A.get_parent().get_resname(),  # Residue name, e.g., "LEU"
            "atom1_residue_num": atom_A.get_parent().get_id()[1],    # Residue sequence number (e.g., 132)
            "atom1_insertion_code": atom_A.get_parent().get_id()[2],    # Insertion code, if present
            "atom1_chain_id": atom_A.get_parent().get_parent().id,  # Chain ID, e.g., "F1"
            "atom2_name": atom_B.get_name(),             # Atom name, e.g., "CG2"
            "atom2_residue": atom_B.get_parent().get_resname(),  # Residue name, e.g., "LEU"
            "atom2_residue_num": atom_B.get_parent().get_id()[1],    # Residue sequence number (e.g., 132)
            "atom2_insertion_code": atom_B.get_parent().get_id()[2],    # Insertion code, if present
            "atom2_chain_id": atom_B.get_parent().get_parent().id,  # Chain ID, e.g., "F1"
            "distance": dist,
            "dock": dock_label
        })
    return sorted(distances, key=lambda x: x["distance"])  # return top 10

In [5]:
# --- Folder structure ---
base_folder = "boltz_results_il17/predictions"

# --- MMCIF parser ---
parser = MMCIFParser(QUIET=True)

# --- Collect all distance entries ---
all_distances = []
all_affinities = []
all_confidences = []

# --- Traverse subfolders ---
for subfolder in os.listdir(base_folder):
    subfolder_path = os.path.join(base_folder, subfolder)
    if os.path.isdir(subfolder_path):
        # Find the .cif file in the subfolder
        cif_files = [f for f in os.listdir(subfolder_path) if f.endswith(".cif")]
        if not cif_files:
            print(f"No .cif file in {subfolder_path}")
            continue

        cif_path = os.path.join(subfolder_path, cif_files[0])
        try:
            structure = parser.get_structure(subfolder, cif_path)
            model = structure[0]
            chains = list(model.get_chains())

            if len(chains) < 3:
                print(f"Skipping {subfolder}: fewer than 3 chains found")
                continue

            protein_1 = chains[0]
            protein_2 = chains[1]
            drug = chains[2]

            # Compute and collect distances
            all_distances.extend(get_distances(protein_1, drug, subfolder))
            all_distances.extend(get_distances(protein_2, drug, subfolder))

        except Exception as e:
            print(f"Error processing {cif_path}: {e}")

        affinity_files = [f for f in os.listdir(subfolder_path) if f.startswith("affinity_")]

        if not affinity_files:
            print(f"No affinity file in {subfolder_path}")
            continue

        affinity_path = os.path.join(subfolder_path, affinity_files[0])

        affinity = None
        with open(affinity_path) as affinity_file:
            affinity = json.load(affinity_file)

        if affinity is not None:
            affinity["dock"] = subfolder
            all_affinities.extend([affinity])

        confidence_files = [f for f in os.listdir(subfolder_path) if f.startswith("confidence_")]

        if not confidence_files:
            print(f"No confidence file in {subfolder_path}")
            continue

        confidence_path = os.path.join(subfolder_path, confidence_files[0])

        confidence = None
        with open(confidence_path) as confidence_file:
            confidence = json.load(confidence_file)

        if confidence is not None:
            confidence["dock"] = subfolder
            all_confidences.extend([confidence])


# --- Convert to DataFrame ---
distance_df = pd.DataFrame(all_distances)
affinity_df = pd.DataFrame(all_affinities)
confidence_df = pd.DataFrame(all_confidences)

In [8]:
distance_df_condensed = distance_df.groupby('dock').apply(lambda g: pd.Series({
    'percent_hit': 100 * (((g['atom1_residue_num'].between(61, 69) | g['atom1_residue_num'].between(79, 115)) & (g['distance'] < 4)).sum() / len(g))
})).reset_index()

result_df = distance_df_condensed.merge(affinity_df, on='dock').merge(confidence_df, on='dock')
result_df[['company', 'patent_ref_number', 'dimer']] = result_df['dock'].str.split('_', expand=True)

/tmp/ipykernel_5882/1780998216.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  distance_df_condensed = distance_df.groupby('dock').apply(lambda g: pd.Series({


In [9]:
distance_df.to_csv('distance_df_il17.csv')
result_df.to_csv('result_df_il17.csv')

In [7]:
result_df

NameError: name 'result_df' is not defined